In [37]:
import pandas as pd
from pathlib import Path
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
DATA_FOLDER = Path.cwd().parent / "artifacts" / "data_ingestion"

In [39]:
# Load each CSV, keep DataFrames in memory, and show .info()
csv_files = sorted(DATA_FOLDER.glob("*.csv"))

dataframes = {}
for csv_path in csv_files:
    name = csv_path.stem
    df = pd.read_csv(csv_path)
    dataframes[name] = df
    globals()[f"{name}_df"] = df
    print(f"\n{name}_df")
    df.info()



anime_df
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16206 entries, 0 to 16205
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         16206 non-null  int64 
 1   title      16206 non-null  object
 2   rating     16206 non-null  object
 3   synopsis   16206 non-null  object
 4   ep_bin     16206 non-null  object
 5   dur_bin    16206 non-null  object
 6   era        16206 non-null  object
 7   source_id  16206 non-null  int64 
 8   favorites  16206 non-null  int64 
 9   watching   16206 non-null  int64 
 10  completed  16206 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 1.4+ MB

anime_genres_df
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48164 entries, 0 to 48163
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   anime_id  48164 non-null  int64
 1   genre_id  48164 non-null  int64
dtypes: int64(2)
memory usage: 752.7 KB

anime_prod

In [40]:
user_anime_ratings_df.isnull().sum()


user_id     0
anime_id    0
rating      0
dtype: int64

In [41]:
anime_df.head()

,id,title,rating,synopsis,ep_bin,dur_bin,era,source_id,favorites,watching,completed
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,10,22,1591,13652
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,11,0,12,0
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,10,0,4,336
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,10,0,7,267
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,10,0,16,255


Merging tables for content based recommender

In [42]:
df = anime_df.drop(columns=["favorites", "watching", "completed"])
df.head()

,id,title,rating,synopsis,ep_bin,dur_bin,era,source_id
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,10
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,11
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,10
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,10
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,10


In [43]:
genres_df.head()

,id,genre_name
0,0,Unknown
1,1,Action
2,2,Adventure
3,3,Cars
4,4,Comedy


In [44]:
producers_df.head()

,id,producer_name
0,0,Unknown
1,1,12 Diary Holders
2,2,1st PLACE
3,3,1theK
4,4,3xCube


In [45]:
studios_df.head()

,id,studio_name
0,0,Unknown
1,1,10Gauge
2,2,2:10 AM Animation
3,3,33 Collective
4,4,3xCube


In [46]:
sources_df.head()

,id,source_name
0,0,Unknown
1,1,4-koma manga
2,2,Book
3,3,Card game
4,4,Digital manga


In [47]:
anime_genres_df.head()

,anime_id,genre_id
0,7092,4
1,7092,27
2,7092,29
3,7092,36
4,33986,4


In [48]:
anime_producers_df.head()

,anime_id,producer_id
0,7092,0
1,33986,492
2,2920,0
3,1719,584
4,6293,514


In [49]:
anime_studios_df.head()

,anime_id,studio_id
0,7092,373
1,33986,0
2,2920,307
3,1719,320
4,6293,0


In [50]:
genres_agg = (
    anime_genres_df
    .merge(genres_df.rename(columns={"id": "genre_id"}), on="genre_id")
    .groupby("anime_id")["genre_name"]
    .apply(list)
    .reset_index(name="genres")
)

producers_agg = (
    anime_producers_df
    .merge(producers_df.rename(columns={"id": "producer_id"}), on="producer_id")
    .groupby("anime_id")["producer_name"]
    .apply(list)
    .reset_index(name="producers")
)

studios_agg = (
    anime_studios_df
    .merge(studios_df.rename(columns={"id": "studio_id"}), on="studio_id")
    .groupby("anime_id")["studio_name"]
    .apply(list)
    .reset_index(name="studios")
)

final_df = (
    df.rename(columns={"id": "anime_id"})
    .merge(
        sources_df.rename(columns={"id": "source_id", "source_name": "source"}),
        on="source_id",
        how="left",
    )
    .merge(genres_agg, on="anime_id", how="left")
    .merge(producers_agg, on="anime_id", how="left")
    .merge(studios_agg, on="anime_id", how="left")
    .drop(columns=["source_id"])
)

final_df.head()


,anime_id,title,rating,synopsis,ep_bin,dur_bin,era,source,genres,producers,studios
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,Original,"[Action, Drama, Mecha, Sci-Fi]","[Dentsu, AT-X, Ultra Super Pictures, Sony Musi...",[SANZIGEN]
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,Other,"[Comedy, Kids, Slice of Life]",[Unknown],[Unknown]
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,Original,[Dementia],[Unknown],[Unknown]
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,Original,[Dementia],[Studio Zero],[Unknown]
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,Original,"[Comedy, Magic]",[Unknown],[MooGoo]


In [52]:
len(final_df)

16206

In [53]:
for col in ["genres", "producers", "studios"]:
    final_df[col] = final_df[col].apply(
        lambda lst: [x for x in lst if x != "Unknown"] if isinstance(lst, list) else []
    )

final_df["source"] = final_df["source"].replace("Unknown", "")

final_df.head()


,anime_id,title,rating,synopsis,ep_bin,dur_bin,era,source,genres,producers,studios
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,Original,"[Action, Drama, Mecha, Sci-Fi]","[Dentsu, AT-X, Ultra Super Pictures, Sony Musi...",[SANZIGEN]
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,Other,"[Comedy, Kids, Slice of Life]",[],[]
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,Original,[Dementia],[],[]
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,Original,[Dementia],[Studio Zero],[]
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,Original,"[Comedy, Magic]",[],[MooGoo]


User_based_df

In [55]:
users_df.head()

,user_id,email,username,password
0,2374,2374@gmail.com,test_2374,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
1,2375,2375@gmail.com,test_2375,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
2,2376,2376@gmail.com,test_2376,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
3,2377,2377@gmail.com,test_2377,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
4,2378,2378@gmail.com,test_2378,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...


In [56]:
user_anime_ratings_df.head()

,user_id,anime_id,rating
0,204,35756,5
1,204,8142,7
2,204,28833,4
3,204,28999,4
4,204,30485,4


In [60]:
u = set()

for user in user_anime_ratings_df["user_id"].unique():
    if user not in users_df["user_id"].values:
        u.add(user)
u

set()